# 02 - Microsoft GraphRAG Quickstart

**Goal**: Run the full GraphRAG pipeline (indexing + querying) on sample text.

**Prerequisites**:
- `pip install graphrag`
- OpenAI API key in `.env` (or Ollama running locally)

**Time**: ~20 minutes (indexing may take 2-5 min depending on LLM)

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

PROJECT_ROOT = Path("..").resolve()
GRAPHRAG_DIR = PROJECT_ROOT / "graphrag_workspace"

print(f"Project root: {PROJECT_ROOT}")
print(f"GraphRAG workspace: {GRAPHRAG_DIR}")
print(f"API key set: {'Yes' if os.getenv('OPENAI_API_KEY') else 'No -- set it in .env!'}")

## Step 1: Initialize GraphRAG Workspace

Run this in your terminal from the project root:

```bash
mkdir -p graphrag_workspace && cd graphrag_workspace
graphrag init
```

This creates:
- `settings.yaml` — pipeline configuration
- `.env` — API keys
- `input/` — place your text files here

## Step 2: Add Sample Text

For your first run, use a short text. Below we create a sample about ML history.

In [ ]:
# Create sample input text
input_dir = GRAPHRAG_DIR / "input"
input_dir.mkdir(parents=True, exist_ok=True)

sample_text = """
The field of deep learning was revolutionized by several key breakthroughs. In 2012, 
Geoffrey Hinton and his students Alex Krizhevsky and Ilya Sutskever developed AlexNet, 
which won the ImageNet competition by a large margin. This demonstrated that deep 
convolutional neural networks could dramatically outperform traditional computer vision methods.

Yann LeCun, working at Bell Labs and later Facebook AI Research (FAIR), had pioneered 
convolutional neural networks in the 1990s with LeNet, which was used for digit recognition. 
His work laid the foundation for AlexNet and all subsequent CNN architectures.

Yoshua Bengio, based at the University of Montreal and Mila, made fundamental contributions 
to sequence modeling and attention mechanisms. His research group explored neural machine 
translation and the attention mechanism that would later inspire the Transformer architecture.

In 2017, Ashish Vaswani and colleagues at Google Brain published "Attention Is All You Need," 
introducing the Transformer architecture. This replaced recurrent neural networks with 
self-attention mechanisms, enabling much more efficient parallel training. The Transformer 
became the foundation for BERT (developed by Jacob Devlin at Google) and GPT (developed by 
Alec Radford at OpenAI).

Ian Goodfellow, while a student of Yoshua Bengio at the University of Montreal, invented 
Generative Adversarial Networks (GANs) in 2014. GANs use two competing neural networks — 
a generator and a discriminator — to create realistic synthetic data. This approach 
revolutionized generative modeling and found applications in image synthesis, style transfer, 
and data augmentation.

The three pioneers — Hinton, LeCun, and Bengio — were jointly awarded the Turing Award in 
2018 for their contributions to deep learning. They are often referred to as the "Godfathers 
of Deep Learning." Their work at institutions including the University of Toronto, NYU, and 
the University of Montreal created the foundation for modern artificial intelligence.
"""

(input_dir / "ml_history.txt").write_text(sample_text)
print(f"Sample text written to {input_dir / 'ml_history.txt'}")
print(f"Text length: {len(sample_text)} characters")

## Step 3: Run Indexing

Run this in your terminal:

```bash
cd graphrag_workspace
graphrag index
```

This will:
1. Split text into chunks
2. Extract entities and relationships using the LLM
3. Build a knowledge graph
4. Detect communities (Leiden algorithm)
5. Generate community summaries

**Cost note**: For this small text with gpt-4o-mini, expect ~$0.01-0.05.

## Step 4: Query — Local vs Global

After indexing completes, try both query modes:

```bash
# Local search: entity-focused, specific questions
graphrag query --method local "What did Geoffrey Hinton contribute to deep learning?"

# Global search: theme-focused, big picture questions  
graphrag query --method global "What are the main themes and breakthroughs in this text?"
```

### Understanding the difference:
- **Local search** finds the entity "Geoffrey Hinton" in the graph, retrieves its neighborhood (connected papers, institutions, co-authors), and generates an answer from that local context.
- **Global search** uses community summaries to answer broad questions by map-reducing across all community descriptions.

## Step 5: Explore the Generated Graph

After indexing, GraphRAG stores results in `output/`. Let's inspect them.

In [ ]:
import pandas as pd

output_dir = GRAPHRAG_DIR / "output"

# Check what files were generated
if output_dir.exists():
    for f in sorted(output_dir.rglob("*.parquet")):
        print(f"  {f.relative_to(output_dir)}")
else:
    print("No output yet — run 'graphrag index' first!")

In [ ]:
# Load and inspect entities (after indexing)
entities_file = list(output_dir.rglob("*entities*.parquet"))
if entities_file:
    entities = pd.read_parquet(entities_file[0])
    print(f"Found {len(entities)} entities:")
    print(entities[["title", "type"]].to_string())
else:
    print("Run 'graphrag index' first to generate entities.")

In [ ]:
# Load and inspect relationships (after indexing)
rels_file = list(output_dir.rglob("*relationships*.parquet"))
if rels_file:
    rels = pd.read_parquet(rels_file[0])
    print(f"Found {len(rels)} relationships:")
    print(rels[["source", "target", "description"]].head(20).to_string())
else:
    print("Run 'graphrag index' first to generate relationships.")

## Key Observations

After running this notebook, you should understand:

1. **GraphRAG automatically extracts entities** (people, organizations, concepts) and their relationships from raw text
2. **Local search** is great for specific questions about known entities
3. **Global search** is great for summarization and theme discovery
4. **The pipeline is LLM-intensive** — entity extraction calls the LLM many times

### Next: 03_build_your_project.ipynb
Apply this to your chosen dataset (ArXiv papers, movies, etc.)